In [2]:
!pip install geemap earthengine-api -q

import ee
import geemap
import numpy as np
import matplotlib.pyplot as plt

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 19.9 MB/s eta 0:00:00


In [3]:
import ee
import geemap
import pandas as pd

ee.Authenticate()
ee.Initialize(project='ee-pakjira')

In [4]:
chonburi = ee.FeatureCollection("FAO/GAUL/2015/level1") \
    .filter(ee.Filter.eq('ADM1_NAME', 'Chonburi'))

Map = geemap.Map()
Map.centerObject(chonburi, 10)
Map.addLayer(chonburi, {}, 'Chonburi')
Map

Map(center=[13.190308961713516, 101.20230008486341], controls=(WidgetControl(options=['position', 'transparent…

In [5]:
start = '2023-01-01'
end = '2023-12-31'

In [6]:
lst = ee.ImageCollection("MODIS/061/MOD11A2") \
    .filterDate(start, end) \
    .select('LST_Day_1km') \
    .mean() \
    .multiply(0.02) \
    .subtract(273.15) \
    .clip(chonburi)

In [7]:
ndvi = ee.ImageCollection("MODIS/061/MOD13A2") \
    .filterDate(start, end) \
    .select('NDVI') \
    .mean() \
    .multiply(0.0001) \
    .clip(chonburi)

In [8]:
landsat = ee.ImageCollection("LANDSAT/LC08/C02/T1_L2") \
    .filterDate(start, end) \
    .filterBounds(chonburi)

def calc_ndbi(img):
    nir = img.select('SR_B5')
    swir = img.select('SR_B6')
    return swir.subtract(nir).divide(swir.add(nir)).rename('NDBI')

ndbi = landsat.map(calc_ndbi).mean().clip(chonburi)

In [9]:
water = ee.Image("JRC/GSW1_4/GlobalSurfaceWater") \
    .select('occurrence') \
    .gt(50)

distance = water.Not().fastDistanceTransform(30).sqrt() \
    .multiply(30) \
    .clip(chonburi)

In [10]:
def normalize(img):
    stats = img.reduceRegion(
        reducer=ee.Reducer.minMax(),
        geometry=chonburi,
        scale=1000,
        maxPixels=1e13
    )

    min_val = ee.Number(stats.values().get(0))
    max_val = ee.Number(stats.values().get(1))

    return img.subtract(min_val).divide(max_val.subtract(min_val))

In [11]:
# Positive
lst_n = normalize(lst)
ndbi_n = normalize(ndbi)

# Negative (กลับค่า)
ndvi_n = normalize(ndvi).multiply(-1).add(1)
dist_n = normalize(distance).multiply(-1).add(1)

In [12]:
uhs = (lst_n.multiply(0.4)
       .add(ndvi_n.multiply(0.2))
       .add(ndbi_n.multiply(0.3))
       .add(dist_n.multiply(0.1)))

In [13]:
vis = {
    'min': 0,
    'max': 1,
    'palette': ['green', 'yellow', 'orange', 'red']
}

Map.addLayer(uhs, vis, 'Urban Heat Stress')

class_vis = {
    'min': 1,
    'max': 5,
    'palette': ['green','lightgreen','yellow','orange','red']
}

uhs_class = uhs.expression(
    " (b('LST_Day_1km') <= 0.2) ? 1" +
    ": (b('LST_Day_1km') <= 0.4) ? 2" +
    ": (b('LST_Day_1km') <= 0.6) ? 3" +
    ": (b('LST_Day_1km') <= 0.8) ? 4" +
    ": 5",
    {'LST_Day_1km': uhs}
)

Map.addLayer(uhs_class, class_vis, 'UHS Classes')

Map

Map(bottom=121681.0, center=[13.190308961713516, 101.20230008486341], controls=(WidgetControl(options=['positi…

In [14]:
task = ee.batch.Export.image.toDrive(
    image=uhs,
    description='UHS_chonburi',
    scale=30,
    region=chonburi.geometry(),
    maxPixels=1e13
)
task.start()

In [15]:
# =======================================================
# Sensitivity Analysis: Heat (LST) ±20%
# =======================================================

# ------------------------------
# Baseline (น้ำหนักปกติ)
# ------------------------------
uhs_baseline = (lst_n.multiply(0.40)
    .add(ndvi_n.multiply(0.20))
    .add(ndbi_n.multiply(0.30))
    .add(dist_n.multiply(0.10))
    .rename('UHS'))

# ------------------------------
# Heat +20%  (0.40 -> 0.48)
# ------------------------------
uhs_plus_20 = (lst_n.multiply(0.48)
    .add(ndvi_n.multiply(0.18))
    .add(ndbi_n.multiply(0.28))
    .add(dist_n.multiply(0.06))
    .rename('UHS'))

# ------------------------------
# Heat -20%  (0.40 -> 0.32)
# ------------------------------
uhs_minus_20 = (lst_n.multiply(0.32)
    .add(ndvi_n.multiply(0.22))
    .add(ndbi_n.multiply(0.30))
    .add(dist_n.multiply(0.16))
    .rename('UHS'))

In [16]:
# =======================================================
# Map
# =======================================================
Map = geemap.Map()
Map.centerObject(chonburi, 10)

# ------------------------------
# Risk Layers
# ------------------------------
vis = {'min':0, 'max':1}

Map.addLayer(uhs_baseline, {**vis, 'palette':['green','yellow','red']}, 'Baseline UHS')
Map.addLayer(uhs_plus_20, {**vis, 'palette':['white','red']}, 'Heat +20%')
Map.addLayer(uhs_minus_20, {**vis, 'palette':['white','blue']}, 'Heat -20%')

In [17]:
# =======================================================
# Sensitivity Detection
# =======================================================

diff_plus  = uhs_plus_20.subtract(uhs_baseline).abs()
diff_minus = uhs_minus_20.subtract(uhs_baseline).abs()

robust = diff_plus.lt(0.05).And(diff_minus.lt(0.05))
sensitive = robust.Not()

# ------------------------------
# Add Layers
# ------------------------------
Map.addLayer(robust.updateMask(robust), {'palette':['green']}, 'Robust Areas')
Map.addLayer(sensitive.updateMask(sensitive), {'palette':['purple']}, 'Sensitive Areas')

Map

Map(center=[13.190308961713516, 101.20230008486341], controls=(WidgetControl(options=['position', 'transparent…

In [18]:
# -----------------------------
# 1. ROI
# -----------------------------
roi = ee.FeatureCollection('FAO/GAUL/2015/level1') \
    .filter(ee.Filter.eq('ADM1_NAME', 'Chonburi'))

# -----------------------------
# 2. LST (ใช้ max เพื่อจับความร้อนจริง)
# -----------------------------
lst_val = ee.ImageCollection("MODIS/061/MOD11A2") \
    .filterDate('2023-02-01', '2023-05-31') \
    .select('LST_Day_1km') \
    .max() \
    .multiply(0.02) \
    .subtract(273.15) \
    .clip(roi)

# -----------------------------
# 3. Heat Mask (ไม่เข้มเกิน)
# -----------------------------
heat_mask = lst_val.gt(33).selfMask()   # 🔥 ปรับตรงนี้

# -----------------------------
# 4. Model Threshold (กลางๆ)
# -----------------------------
high_risk = uhs.gte(0.35)                # 🔥 ปรับตรงนี้

# -----------------------------
# 5. Correct Prediction
# -----------------------------
correct_heat = heat_mask.updateMask(high_risk)

# -----------------------------
# 6. Accuracy
# -----------------------------
total_heat = heat_mask.reduceRegion(
    reducer=ee.Reducer.sum(),
    geometry=roi,
    scale=1000,
    maxPixels=1e13
).values().get(0).getInfo()

correct_heat_val = correct_heat.reduceRegion(
    reducer=ee.Reducer.sum(),
    geometry=roi,
    scale=1000,
    maxPixels=1e13
).values().get(0).getInfo()

accuracy = 0 if total_heat == 0 else (correct_heat_val / total_heat) * 100

print("\n========== VALIDATION RESULT ==========")
print(f"Total Hot Pixels (>33°C): {total_heat}")
print(f"Correctly Predicted Pixels: {correct_heat_val}")
print(f"Accuracy (%): {accuracy:.2f}")
print("======================================")

# -----------------------------
# 7. Visualization
# -----------------------------
Map = geemap.Map()
Map.centerObject(roi, 10)

Map.addLayer(lst_val, {'min':25,'max':40,'palette':['blue','yellow','red']}, 'LST')
Map.addLayer(heat_mask, {'palette':'black'}, 'Actual Heat (>33°C)')
Map.addLayer(correct_heat, {'palette':'blue'}, 'Correct Prediction')
Map.addLayer(roi, {'color':'red'}, 'Chonburi')

Map


========== VALIDATION RESULT ==========
Total Hot Pixels (>33°C): 4268.53725490196
Correctly Predicted Pixels: 3436.8745098039212
Accuracy (%): 80.52


Map(center=[13.190308961713562, 101.20230008486342], controls=(WidgetControl(options=['position', 'transparent…